# 04 – Feature Engineering

**Mục tiêu:** tạo thêm đặc trưng từ dữ liệu đã tiền xử lý (bước 03) và kiểm tra bằng `GroupKFold` (theo `group_id`) xem đặc trưng mới có thực sự cải thiện macro-F1 hay không.

Các nhóm đặc trưng:
1. **Nghệ sĩ**: tần suất xuất hiện của `Artist Name` (frequency encoding, không dùng nhãn nên không rò rỉ).
2. **Tên bài hát**: độ dài, số từ, có ngoặc, có `feat/ft/with`.
3. **Kết hợp**: tích / hiệu của các đặc trưng âm thanh có liên quan (energy × loudness, energy − acousticness, ...).

> Lưu ý: tên nghệ sĩ/bài hát đã bị bỏ ở bước 03 nên ở đây đọc lại từ file gốc và ghép theo `Id`.

In [9]:
RUNNER = "Khudz123"   # <-- đổi thành tên người chạy

import sys, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score

warnings.filterwarnings("ignore")
# Tự tìm thư mục gốc "Challenge 2" (nơi có DATA_CHALLENGE2) bằng cách đi ngược lên từ thư mục hiện tại,
# nên notebook chạy đúng dù đặt trong notebooks/ hay notebooks/04_Feature_Engineering/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "DATA_CHALLENGE2").exists())
NB_DIR = ROOT / "notebooks"          # nơi chứa logger_utils.py và preprocessing.py
sys.path.insert(0, str(NB_DIR))
from logger_utils import ExperimentLogger

DATA_DIR = ROOT / "DATA_CHALLENGE2"
PROC_DIR = DATA_DIR / "processed"
LOG_DIR  = ROOT / "logs"
SEED = 42

log = ExperimentLogger(
    runner=RUNNER,
    stage="04_feature_engineering",
    notebook="04_Feature_Engineering.ipynb",
    log_dir=str(LOG_DIR),
    goal="Tạo đặc trưng mới (artist freq, tên bài hát, đặc trưng kết hợp) và đánh giá bằng GroupKFold macro-F1",
)

## 1. Nạp dữ liệu

In [10]:
train_p = pd.read_csv(PROC_DIR / "train_processed.csv")
test_p  = pd.read_csv(PROC_DIR / "test_processed.csv")
train_raw = pd.read_csv(DATA_DIR / "train (1).csv", usecols=["Id", "Artist Name", "Track Name"])
test_raw  = pd.read_csv(DATA_DIR / "test (2).csv",  usecols=["Id", "Artist Name", "Track Name"])

# ghép tên nghệ sĩ / bài hát vào bảng đã xử lý (theo Id, giữ nguyên thứ tự dòng)
train = train_p.merge(train_raw, on="Id", how="left", validate="one_to_one")
test  = test_p.merge(test_raw,  on="Id", how="left", validate="one_to_one")
assert len(train) == len(train_p) and len(test) == len(test_p)

for d in (train, test):
    d["Artist Name"] = d["Artist Name"].fillna("").astype(str)
    d["Track Name"]  = d["Track Name"].fillna("").astype(str)

BASE_COLS = [c for c in train_p.columns if c not in ("Id", "group_id", "Class")]
print("Train:", train.shape, "| Test:", test.shape, "| số đặc trưng gốc:", len(BASE_COLS))
log.step("Nạp dữ liệu", train_shape=train.shape, test_shape=test.shape, n_base_features=len(BASE_COLS))

Train: (14396, 37) | Test: (3600, 35) | số đặc trưng gốc: 32


## 2. Tạo đặc trưng

- Tần suất nghệ sĩ được đếm trên **train + test gộp** (chỉ đếm tên, không dùng nhãn → không rò rỉ nhãn).
- Các đặc trưng kết hợp được tính từ các cột liên tục **đã chuẩn hoá** (z-score), nên tích/hiệu là tương tác giữa các độ lệch so với trung bình.

In [11]:
CONT = ["Popularity", "danceability", "energy", "loudness", "speechiness",
        "acousticness", "instrumentalness", "liveness", "valence", "tempo", "duration_min"]
missing = [c for c in CONT if c not in train.columns]
assert not missing, f"Thiếu cột liên tục: {missing}"

all_artists = pd.concat([train["Artist Name"], test["Artist Name"]])
artist_counts = all_artists.value_counts()

def add_features(df):
    out = df.copy()
    # --- nghệ sĩ ---
    out["artist_freq"] = np.log1p(out["Artist Name"].map(artist_counts).fillna(1))
    out["artist_unknown"] = out["Artist Name"].str.strip().str.lower().isin(["", "unknown", "various artists"]).astype(int)
    # --- tên bài hát ---
    t = out["Track Name"]
    out["track_len"]   = t.str.len()
    out["track_words"] = t.str.split().str.len().fillna(0)
    out["track_paren"] = t.str.contains(r"[\(\[]", regex=True).astype(int)
    out["track_feat"]  = t.str.contains(r"\b(?:feat|ft|with)\b\.?", case=False, regex=True).astype(int)
    # --- kết hợp ---
    out["energy_x_loud"]       = out["energy"] * out["loudness"]
    out["energy_minus_acoust"] = out["energy"] - out["acousticness"]
    out["loud_minus_acoust"]   = out["loudness"] - out["acousticness"]
    out["dance_x_valence"]     = out["danceability"] * out["valence"]
    out["speech_x_instr"]      = out["speechiness"] * out["instrumentalness"]
    out["acoust_x_instr"]      = out["acousticness"] * out["instrumentalness"]
    out["tempo_x_energy"]      = out["tempo"] * out["energy"]
    out["dance_x_energy"]      = out["danceability"] * out["energy"]
    return out

train_fe = add_features(train)
test_fe  = add_features(test)

ARTIST_FE = ["artist_freq", "artist_unknown"]
TRACK_FE  = ["track_len", "track_words", "track_paren", "track_feat"]
COMBO_FE  = ["energy_x_loud", "energy_minus_acoust", "loud_minus_acoust", "dance_x_valence",
             "speech_x_instr", "acoust_x_instr", "tempo_x_energy", "dance_x_energy"]
NEW_COLS = ARTIST_FE + TRACK_FE + COMBO_FE

# track_len / track_words chưa được chuẩn hoá: chuẩn hoá bằng thống kê của TRAIN
for c in ["track_len", "track_words", "artist_freq"]:
    mu, sd = train_fe[c].mean(), train_fe[c].std() + 1e-9
    train_fe[c] = (train_fe[c] - mu) / sd
    test_fe[c]  = (test_fe[c]  - mu) / sd

print("Số đặc trưng mới:", len(NEW_COLS))
display(train_fe[NEW_COLS].describe().T[["mean", "std", "min", "max"]].round(3))
log.step("Tạo đặc trưng mới", artist=ARTIST_FE, track=TRACK_FE, combo=COMBO_FE)

Số đặc trưng mới: 14


,mean,std,min,max
artist_freq,0.000,1.000,-0.939,3.455
artist_unknown,0.000,0.008,0.000,1.000
track_len,0.000,1.000,-1.402,8.536
track_words,0.000,1.000,-0.963,9.083
track_paren,0.075,0.263,0.000,1.000
track_feat,0.052,0.223,0.000,1.000
energy_x_loud,0.778,1.561,-1.674,10.745
energy_minus_acoust,0.000,1.869,-5.199,2.215
loud_minus_acoust,0.000,1.801,-6.244,2.262
dance_x_valence,0.445,1.019,-3.794,5.614


## 3. Đánh giá bằng GroupKFold

So sánh macro-F1 (Random Forest, `class_weight="balanced"`) khi thêm từng nhóm đặc trưng. Chia fold theo `group_id` để cùng một bài hát không nằm ở cả train và validation.

> Chênh lệch nhỏ (< ~0,005) nằm trong nhiễu của CV, đừng kết luận vội.

In [12]:
y = train_fe["Class"].values
groups = train_fe["group_id"].values
gkf = GroupKFold(n_splits=5)

def cv_score(cols, n_estimators=300):
    f1s, accs = [], []
    X = train_fe[cols].values
    for tr, va in gkf.split(X, y, groups):
        rf = RandomForestClassifier(n_estimators=n_estimators, class_weight="balanced",
                                    n_jobs=-1, random_state=SEED)
        rf.fit(X[tr], y[tr])
        pred = rf.predict(X[va])
        f1s.append(f1_score(y[va], pred, average="macro"))
        accs.append(accuracy_score(y[va], pred))
    return np.mean(f1s), np.std(f1s), np.mean(accs)

experiments = {
    "0. Gốc (32 đặc trưng)":        BASE_COLS,
    "1. Gốc + Artist":              BASE_COLS + ARTIST_FE,
    "2. Gốc + Tên bài hát":         BASE_COLS + TRACK_FE,
    "3. Gốc + Kết hợp":             BASE_COLS + COMBO_FE,
    "4. Gốc + Tất cả":              BASE_COLS + NEW_COLS,
}
rows = []
for name, cols in experiments.items():
    m, s, a = cv_score(cols)
    rows.append({"thí nghiệm": name, "n_features": len(cols), "macro_F1": round(m, 4), "std": round(s, 4), "accuracy": round(a, 4)})
    print(f"{name:28s} macro-F1 = {m:.4f} ± {s:.4f} | acc = {a:.4f}")

results = pd.DataFrame(rows)
display(results)
log.step("So sánh GroupKFold (RF, 5 fold)", results=results.to_dict("records"))

0. Gốc (32 đặc trưng)        macro-F1 = 0.5745 ± 0.0043 | acc = 0.5499
1. Gốc + Artist              macro-F1 = 0.6005 ± 0.0071 | acc = 0.5699
2. Gốc + Tên bài hát         macro-F1 = 0.5855 ± 0.0048 | acc = 0.5565
3. Gốc + Kết hợp             macro-F1 = 0.5681 ± 0.0084 | acc = 0.5467
4. Gốc + Tất cả              macro-F1 = 0.5907 ± 0.0080 | acc = 0.5640


,thí nghiệm,n_features,macro_F1,std,accuracy
0,0. Gốc (32 đặc trưng),32,0.5745,0.0043,0.5499
1,1. Gốc + Artist,34,0.6005,0.0071,0.5699
2,2. Gốc + Tên bài hát,36,0.5855,0.0048,0.5565
3,3. Gốc + Kết hợp,40,0.5681,0.0084,0.5467
4,4. Gốc + Tất cả,46,0.5907,0.0080,0.5640


## 4. Mức quan trọng của đặc trưng mới

In [13]:
ALL_COLS = BASE_COLS + NEW_COLS
rf = RandomForestClassifier(n_estimators=500, class_weight="balanced", n_jobs=-1, random_state=SEED)
rf.fit(train_fe[ALL_COLS].values, y)
imp = pd.Series(rf.feature_importances_, index=ALL_COLS).sort_values(ascending=False)
print("Top 15 đặc trưng:")
display(imp.head(15).round(4))
print("Đặc trưng mới, xếp hạng trong tổng", len(ALL_COLS), ":")
rank = imp.rank(ascending=False).astype(int)
display(pd.DataFrame({"importance": imp[NEW_COLS].round(4), "rank": rank[NEW_COLS]}).sort_values("rank"))
log.step("Feature importance (RF 500 cây)", top5=imp.head(5).round(4).to_dict())

Top 15 đặc trưng:


dur_was_minutes        0.0928
acousticness           0.0547
Popularity             0.0530
energy_minus_acoust    0.0519
speechiness            0.0514
acoust_x_instr         0.0492
loud_minus_acoust      0.0465
valence                0.0456
speech_x_instr         0.0454
energy                 0.0420
duration_min           0.0410
instrumentalness       0.0401
danceability           0.0391
energy_x_loud          0.0355
loudness               0.0339
dtype: float64

Đặc trưng mới, xếp hạng trong tổng 46 :


,importance,rank
energy_minus_acoust,0.0519,4
acoust_x_instr,0.0492,6
loud_minus_acoust,0.0465,7
speech_x_instr,0.0454,9
energy_x_loud,0.0355,14
artist_freq,0.0337,16
dance_x_valence,0.0289,17
dance_x_energy,0.0284,18
track_len,0.0261,20
tempo_x_energy,0.0248,22


## 5. Chọn bộ đặc trưng và lưu

Chọn bộ có macro-F1 cao nhất ở bảng trên. Mặc định bên dưới lấy theo kết quả CV; có thể ghi đè `FINAL_COLS` bằng tay nếu chênh lệch nằm trong nhiễu và bạn muốn giữ mô hình đơn giản.

In [14]:
best_name = results.sort_values("macro_F1", ascending=False).iloc[0]["thí nghiệm"]
FINAL_COLS = experiments[best_name]
print("Bộ đặc trưng được chọn:", best_name, "->", len(FINAL_COLS), "cột")

train_out = train_fe[["Id"] + FINAL_COLS + ["group_id", "Class"]]
test_out  = test_fe[["Id"] + FINAL_COLS]

train_out.to_csv(PROC_DIR / "train_fe.csv", index=False)
test_out.to_csv(PROC_DIR / "test_fe.csv", index=False)
pd.Series(FINAL_COLS, name="feature").to_csv(PROC_DIR / "feature_cols_fe.csv", index=False)
print("Đã lưu:", PROC_DIR / "train_fe.csv", train_out.shape, "|", PROC_DIR / "test_fe.csv", test_out.shape)
log.step("Lưu kết quả", chosen=best_name, n_features=len(FINAL_COLS),
         train_fe=train_out.shape, test_fe=test_out.shape)

Bộ đặc trưng được chọn: 1. Gốc + Artist -> 34 cột
Đã lưu: d:\May Hoc\CK_ML\MAY_HOC_NHOM\Challenge\Challenge 2\DATA_CHALLENGE2\processed\train_fe.csv (14396, 37) | d:\May Hoc\CK_ML\MAY_HOC_NHOM\Challenge\Challenge 2\DATA_CHALLENGE2\processed\test_fe.csv (3600, 35)


In [15]:
log.save(notes=(
    f"Bộ đặc trưng chọn: {best_name} ({len(FINAL_COLS)} cột). "
    "Frequency encoding Artist đếm trên train+test (không dùng nhãn). "
    "Cột liên tục mới (track_len, track_words, artist_freq) chuẩn hoá bằng thống kê của train. "
    "Đã lưu processed/train_fe.csv, test_fe.csv, feature_cols_fe.csv."
))

✅ Đã ghi nhật ký: experiment_log.csv (thêm 1 dòng) và 04_feature_engineering_20261112_104619_Khudz123.md


WindowsPath('d:/May Hoc/CK_ML/MAY_HOC_NHOM/Challenge/Challenge 2/logs/04_feature_engineering_20261112_104619_Khudz123.md')